# Preprocessing Pipeline — {domain}

**Stage 2 จาก 2** — รับ star schema ที่ stage ELT push ขึ้น Hugging Face ไว้
แล้วแปลงให้พร้อมเทรนโมเดล จากนั้น push เป็น dataset ชุดใหม่

| ขั้น | ทำอะไร |
|------|--------|
| 1 | โหลด star schema จาก HF (ถ้ายังไม่ push จะใช้ไฟล์ในเครื่องแทน) |
| 2 | ล้างข้อความ — ลบ HTML tag/entity, ยุบช่องว่าง |
| 3 | กรองรีวิวที่สั้นเกินไปหลังล้าง |
| 4 | สร้าง label (`sentiment`) + feature ของข้อความ |
| 5 | รวม feature จาก dimension (ราคา, แบรนด์, พฤติกรรมผู้รีวิว, สถิติสินค้า) |
| 6 | แบ่ง train/val/test **ตามเวลา** กัน data leakage |
| 7 | ตรวจคุณภาพ + export Parquet |
| 8 | push ขึ้น Hugging Face |

> ใช้ DuckDB เป็นตัวประมวลผลหลักแทน pandas เพราะข้อมูลระดับหลายล้านแถวที่มี
> ข้อความยาว ๆ ถ้าโหลดเข้า pandas ทั้งก้อนจะกิน RAM หนัก — DuckDB ทำงานบน
> Parquet ได้โดยตรงและ spill ลงดิสก์เองเมื่อหน่วยความจำไม่พอ
> ส่วน pandas ใช้ดูผลลัพธ์ระหว่างทาง

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

# หา project root ไม่ว่าจะรัน notebook จาก root หรือจากโฟลเดอร์ preprocessing/
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from elt.common import human_bytes, load_config  # noqa: E402

config = load_config()
PREP = config["preprocessing"]

con = duckdb.connect()                      # in-memory: อ่าน/เขียน Parquet โดยตรง
con.execute(f"SET memory_limit = '{config['elt']['memory_limit']}'")
con.execute(f"SET threads = {config['elt']['threads']}")

print("domain:    ", config["domain"])
print("categories:", ", ".join(config["categories"]))
print("ELT repo:  ", config["elt"]["repo_id"])
print("output repo:", PREP["repo_id"])

## 1. โหลด star schema จาก stage ELT

ดึงจาก Hugging Face ด้วย `snapshot_download` (รองรับ repo แบบ private อัตโนมัติ
เพราะใช้ token ที่ `huggingface-cli login` เก็บไว้ และ cache ไว้ไม่โหลดซ้ำ)

ถ้ายังไม่ได้ push จะถอยไปใช้ `data/elt_export/` ในเครื่องแทน — pipeline จึงรันได้
ตั้งแต่ก่อน push

In [ ]:
def resolve_elt_source() -> Path:
    """คืน path ของ star schema: จาก HF ถ้า push แล้ว ไม่งั้นใช้ในเครื่อง"""
    local = ROOT / config["elt"]["export_path"]
    repo_id = config["elt"]["repo_id"]

    if "CHANGE_ME" not in repo_id:
        try:
            from huggingface_hub import snapshot_download
            path = Path(snapshot_download(repo_id=repo_id, repo_type="dataset"))
            print(f"แหล่งข้อมูล: Hugging Face — {repo_id}")
            return path
        except Exception as e:
            print(f"โหลดจาก HF ไม่สำเร็จ ({type(e).__name__}: {e})\n→ ใช้ไฟล์ในเครื่องแทน")

    if not local.exists():
        raise FileNotFoundError(
            f"ไม่พบ {local} — รัน `python -m elt.run_elt` ก่อน "
            "หรือตั้ง elt.repo_id ใน config.yaml ให้ชี้ไป dataset ที่ push แล้ว"
        )
    print(f"แหล่งข้อมูล: ไฟล์ในเครื่อง — {local.relative_to(ROOT)}")
    return local


SRC = resolve_elt_source()

# ทำเป็น view คร่อม Parquet — ยังไม่โหลดเข้าหน่วยความจำจนกว่าจะ query
con.execute(f"CREATE VIEW fact_review AS SELECT * FROM '{SRC}/fact_review/**/*.parquet'")
con.execute(f"CREATE VIEW review_text AS SELECT * FROM '{SRC}/review_text/**/*.parquet'")
con.execute(f"CREATE VIEW dim_product AS SELECT * FROM '{SRC}/dim_product.parquet'")
con.execute(f"CREATE VIEW dim_user   AS SELECT * FROM '{SRC}/dim_user.parquet'")
con.execute(f"CREATE VIEW dim_date   AS SELECT * FROM '{SRC}/dim_date.parquet'")

pd.DataFrame(
    [(t, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0])
     for t in ("fact_review", "review_text", "dim_product", "dim_user", "dim_date")],
    columns=["table", "rows"],
)

## 2. ล้างข้อความ

ข้อความรีวิวจริงมี HTML ปนมาเยอะ (`<br />`, `<span>`, `&#34;`, `&amp;`)
ถ้าไม่ล้างก่อน tokenizer จะนับ `br` เป็นคำ และ `&amp;` จะกลายเป็น token ขยะ

`&amp;` ถอดรหัสเป็นลำดับ**สุดท้าย** เพื่อไม่ให้ `&amp;quot;` (ที่ถูก escape สองชั้น)
กลายเป็นเครื่องหมายคำพูดผิด ๆ

In [ ]:
# chr(39) = เครื่องหมาย ' — ใช้แทนการ escape quote ใน SQL ให้อ่านง่ายขึ้น
con.execute(r"""
CREATE OR REPLACE MACRO clean_text(s) AS
    trim(regexp_replace(
        replace(                                     -- &amp; ท้ายสุด
        replace(replace(replace(replace(replace(replace(replace(
            regexp_replace(coalesce(s, ''), '<[^>]{0,200}>', ' ', 'g'),
            '&nbsp;', ' '), '&#34;', '"'), '&quot;', '"'),
            '&#39;', chr(39)), '&apos;', chr(39)), '&lt;', '<'), '&gt;', '>'),
            '&amp;', '&'),
        '\s+', ' ', 'g'));
""")

# ดูตัวอย่างก่อน/หลัง เฉพาะรีวิวที่มี HTML ปนจริง ๆ
con.sql("""
    SELECT review_text AS before, clean_text(review_text) AS after
    FROM review_text
    WHERE review_text LIKE '%<%>%' OR review_text LIKE '%&#%' OR review_text LIKE '%&amp;%'
    LIMIT 5
""").df()

## 3–5. สร้างตารางหลัก: ล้าง → กรอง → feature → label → split

ทำในคำสั่งเดียวเป็นชั้น ๆ (CTE) เพื่อให้ DuckDB วางแผน query ได้ทั้งก้อน:

- **`cleaned`** — ล้างข้อความแล้วกรองรีวิวที่เหลือน้อยกว่าเกณฑ์
- **`product_stats`** — สถิติระดับสินค้า (จำนวนรีวิว, คะแนนเฉลี่ย, วันที่รีวิวแรก)
- **`joined`** — รวม feature จาก dimension + คำนวณ feature ของข้อความ + ตี label + แบ่ง split

In [ ]:
con.execute(f"SET VARIABLE min_words = {PREP['min_words']}")
con.execute(f"SET VARIABLE val_start  = DATE '{PREP['val_start']}'")
con.execute(f"SET VARIABLE test_start = DATE '{PREP['test_start']}'")

con.execute(r"""
CREATE OR REPLACE TABLE reviews_prepared AS
WITH cleaned AS (
    SELECT
        t.review_id,
        t.category,
        clean_text(t.review_title) AS review_title_clean,
        clean_text(t.review_text)  AS review_text_clean
    FROM review_text t
),
sized AS (
    SELECT *,
        length(review_text_clean) AS char_count,
        -- นับคำ = จำนวนช่องว่าง + 1 (ข้อความยุบช่องว่างซ้ำมาแล้ว)
        length(review_text_clean)
            - length(replace(review_text_clean, ' ', '')) + 1 AS word_count
    FROM cleaned
    WHERE length(review_text_clean) > 0
),
filtered AS (
    SELECT * FROM sized WHERE word_count >= getvariable('min_words')
),
product_stats AS (
    SELECT
        product_key,
        count(*)                       AS product_review_count,
        round(avg(rating), 3)          AS product_avg_rating,
        min(CAST(review_ts AS DATE))   AS product_first_review_date
    FROM fact_review
    GROUP BY product_key
)
SELECT
    -- ── ตัวระบุ ────────────────────────────────────────────────
    f.review_id,
    f.category,

    -- ── ข้อความ ────────────────────────────────────────────────
    c.review_title_clean,
    c.review_text_clean,
    trim(CASE WHEN coalesce(c.review_title_clean, '') <> ''
              THEN c.review_title_clean || '. ' ELSE '' END
         || c.review_text_clean) AS text_full,

    -- ── label ─────────────────────────────────────────────────
    f.rating,
    CASE WHEN f.rating <= 2 THEN 'negative'
         WHEN f.rating = 3  THEN 'neutral'
         ELSE                    'positive' END AS sentiment,
    f.is_negative,

    -- ── feature ของข้อความ ─────────────────────────────────────
    c.word_count,
    c.char_count,
    round(
        (c.char_count - (c.word_count - 1)) / nullif(c.word_count, 0), 2
    ) AS avg_word_length,
    c.char_count - length(replace(c.review_text_clean, '!', '')) AS exclamation_count,
    c.char_count - length(replace(c.review_text_clean, '?', '')) AS question_count,
    round(
        length(regexp_replace(c.review_text_clean, '[^A-Z]', '', 'g'))
        / nullif(length(regexp_replace(c.review_text_clean, '[^A-Za-z]', '', 'g')), 0), 4
    ) AS uppercase_ratio,
    length(regexp_replace(c.review_text_clean, '[^0-9]', '', 'g')) AS digit_count,

    -- ── บริบทของรีวิว ──────────────────────────────────────────
    f.verified_purchase,
    f.has_images,
    f.helpful_vote,

    -- ── feature ของสินค้า ──────────────────────────────────────
    p.parent_asin,
    p.store,
    p.price,
    p.price_band,
    ps.product_review_count,
    ps.product_avg_rating,
    round(f.rating - ps.product_avg_rating, 3) AS rating_vs_product_avg,
    date_diff('day', ps.product_first_review_date,
              CAST(f.review_ts AS DATE))       AS days_since_product_first_review,

    -- ── feature ของผู้รีวิว ────────────────────────────────────
    u.lifetime_reviews   AS user_lifetime_reviews,
    u.avg_rating_given   AS user_avg_rating_given,
    u.reviewer_segment,

    -- ── เวลา ──────────────────────────────────────────────────
    CAST(f.review_ts AS DATE) AS review_date,
    year(f.review_ts)         AS review_year,
    month(f.review_ts)        AS review_month,

    -- ── แบ่ง split ตามเวลา ─────────────────────────────────────
    CASE
        WHEN CAST(f.review_ts AS DATE) < getvariable('val_start')  THEN 'train'
        WHEN CAST(f.review_ts AS DATE) < getvariable('test_start') THEN 'val'
        ELSE                                                            'test'
    END AS split
FROM filtered c
JOIN fact_review f USING (review_id)
LEFT JOIN dim_product p  USING (product_key)
LEFT JOIN dim_user u     USING (user_key)
LEFT JOIN product_stats ps USING (product_key)
""")

n_in  = con.execute("SELECT count(*) FROM review_text").fetchone()[0]
n_out = con.execute("SELECT count(*) FROM reviews_prepared").fetchone()[0]
print(f"เข้า {n_in:,} แถว → ออก {n_out:,} แถว "
      f"(ตัดออก {n_in - n_out:,} = {(n_in - n_out) / n_in:.1%} เพราะสั้นเกินเกณฑ์/ข้อความว่าง)")

con.sql("SELECT * FROM reviews_prepared LIMIT 3").df().T

## 5b. ตาราง feature ระดับสินค้า

ชุดนี้ตอบโจทย์ "product performance" โดยตรง — 1 แถว = 1 สินค้า พร้อมสถิติที่
ใช้จัดอันดับ/หาสินค้าขาลงได้ทันที

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE product_features AS
SELECT
    p.product_key,
    p.parent_asin,
    p.category,
    clean_text(p.product_title) AS product_title,
    p.store,
    p.price,
    p.price_band,
    count(*)                                        AS n_reviews,
    round(avg(f.rating), 3)                         AS avg_rating,
    round(stddev_samp(f.rating), 3)                 AS rating_std,
    round(avg(f.is_negative::INT), 4)               AS negative_share,
    round(avg((f.rating IN (1, 5))::INT), 4)        AS polarized_share,
    round(avg(f.verified_purchase::INT), 4)         AS verified_share,
    sum(f.helpful_vote)                             AS total_helpful_votes,
    min(CAST(f.review_ts AS DATE))                  AS first_review_date,
    max(CAST(f.review_ts AS DATE))                  AS last_review_date,
    date_diff('day', min(CAST(f.review_ts AS DATE)),
                     max(CAST(f.review_ts AS DATE))) AS review_span_days
FROM fact_review f
JOIN dim_product p USING (product_key)
GROUP BY ALL
""")

print(con.execute("SELECT count(*) FROM product_features").fetchone()[0], "สินค้า")
con.sql("""
    SELECT product_title, store, price_band, n_reviews, avg_rating, negative_share
    FROM product_features WHERE n_reviews >= 20
    ORDER BY avg_rating DESC LIMIT 5
""").df()

## 6. ตรวจคุณภาพ

เช็คว่าการแบ่ง split ไม่ทับกันตามเวลา, ไม่มีค่าที่ควรมีแล้วหาย, และดูความไม่สมดุลของคลาส
(ซึ่งต้องบอกคนเอาไปเทรนโมเดลต่อ)

In [ ]:
splits = con.sql("""
    SELECT split, count(*) AS n_reviews,
           min(review_date) AS first_date, max(review_date) AS last_date,
           round(avg(rating), 2) AS avg_rating
    FROM reviews_prepared GROUP BY split
    ORDER BY first_date
""").df()
display(splits)

# ช่วงเวลาของแต่ละ split ต้องไม่ทับกัน ไม่งั้นคือ data leakage
ordered = splits.sort_values("first_date")
for a, b in zip(ordered.itertuples(), ordered.iloc[1:].itertuples()):
    assert a.last_date < b.first_date, f"split ทับกัน: {a.split} กับ {b.split}"
print("ช่วงเวลาแต่ละ split ไม่ทับกัน")

# คอลัมน์ที่ห้ามเป็น null
nulls = con.sql("""
    SELECT
        sum(review_id IS NULL)          AS review_id,
        sum(text_full IS NULL)          AS text_full,
        sum(sentiment IS NULL)          AS sentiment,
        sum(rating IS NULL)             AS rating,
        sum(split IS NULL)              AS split,
        sum(product_review_count IS NULL) AS product_review_count
    FROM reviews_prepared
""").df()
assert nulls.sum().sum() == 0, f"เจอ null ในคอลัมน์ที่ห้ามว่าง:\n{nulls.T}"
print("คอลัมน์บังคับ: ไม่มี null")

# review_id ต้องไม่ซ้ำ
dupes = con.execute("""
    SELECT count(*) FROM (SELECT review_id FROM reviews_prepared
                          GROUP BY review_id HAVING count(*) > 1)
""").fetchone()[0]
assert dupes == 0, f"review_id ซ้ำ {dupes} ค่า"
print("review_id: ไม่ซ้ำ")

In [ ]:
# ความไม่สมดุลของคลาส — ต้องเขียนเตือนไว้ใน dataset card
con.sql("""
    SELECT split, sentiment, count(*) AS n,
           round(count(*) * 100.0 / sum(count(*)) OVER (PARTITION BY split), 1) AS pct
    FROM reviews_prepared GROUP BY split, sentiment ORDER BY split, sentiment
""").df().pivot(index="split", columns="sentiment", values="pct")

## 7. Export เป็น Parquet

`reviews/` แบ่ง partition ด้วย `split` — ปลายทางโหลดเฉพาะ train หรือ test ได้เลย

In [ ]:
import json
import shutil

from elt.dataset_card import render_prep_card

out_dir = ROOT / PREP["export_path"]
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True)

con.execute(f"""
    COPY (SELECT * FROM reviews_prepared) TO '{out_dir / "reviews"}'
    (FORMAT PARQUET, PARTITION_BY (split), COMPRESSION ZSTD, OVERWRITE_OR_IGNORE)
""")
con.execute(f"""
    COPY product_features TO '{out_dir / "product_features.parquet"}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

split_counts = dict(con.execute(
    "SELECT split, count(*) FROM reviews_prepared GROUP BY split").fetchall())
stats = {
    "stage": "preprocessing",
    "domain": config["domain"],
    "categories": config["categories"],
    "source_repo": config["elt"]["repo_id"],
    "split_counts": split_counts,
    "product_features": con.execute(
        "SELECT count(*) FROM product_features").fetchone()[0],
    "n_columns": len(con.execute(
        "SELECT * FROM reviews_prepared LIMIT 0").description),
}
(out_dir / "stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))
(out_dir / "README.md").write_text(render_prep_card(config, stats))

for f in sorted(out_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(out_dir)}  ({human_bytes(f.stat().st_size)})")
print(f"\nexport เสร็จ -> {out_dir.relative_to(ROOT)}")

## 8. Push ขึ้น Hugging Face

ตั้ง `preprocessing.repo_id` ใน `config.yaml` และ `huggingface-cli login` ก่อน

เริ่มด้วย `dry_run=True` เพื่อดูรายการไฟล์ก่อน แล้วค่อยเปลี่ยนเป็น `False` เมื่อพอใจ
— repo สร้างเป็น **private** ก่อนเสมอ ค่อยไปกดเปิด public บนเว็บ HF ทีหลัง

In [ ]:
from elt.common import push_folder

push_folder(
    folder=out_dir,
    repo_id=PREP["repo_id"],
    private=PREP["private"],
    message=f"Preprocessed reviews — {config['domain']} (ML ready)",
    dry_run=True,          # <-- เปลี่ยนเป็น False เมื่อพร้อมอัพจริง
)

In [ ]:
con.close()
print("preprocessing pipeline เสร็จเรียบร้อย")